## Evaluate Generative Retrieval Model (T5)

Evaluates the trained T5 seq2seq model with beam search retrieval.

**What this notebook does:**
1. Load trained model checkpoint
2. Beam-search decode top-K Semantic IDs per user
3. Map Semantic IDs to items, filter invalid IDs
4. Compute Recall@K, NDCG@K on val + test splits
5. Compare against paper Table 1 (Toys & Games)

In [1]:
import sys

if "../" not in sys.path:
    sys.path.insert(0, "../")

from functools import partial
from pathlib import Path

import torch
from torch.utils.data import Subset
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, T5ForConditionalGeneration

from tiger.dataset import TigerDataset, custom_collate
from tiger.evaluation import METRICS, compute_metrics
from tiger.utils import get_device, set_seed

In [2]:
# Paths
DATA_DIR = Path("../data/2014/processed")
SPLITS_PATH = DATA_DIR / "splits.parquet"
SEMANTIC_IDS_PATH = Path("../checkpoints/rqvae/semantic_ids.pt")
CHECKPOINT_PATH = Path("../checkpoints/train_eval_test/evaluation_check")

# Evaluation config
BATCH_SIZE = 64  # beam search is memory-heavy; keep modest
AT_K = (5, 10)  # paper evaluates K = 5, 10

SEED = 42

In [3]:
set_seed(SEED)
device = get_device()
print(f"Device: {device}")

2026-09-20 19:44:30.829 | INFO     | tiger.utils:set_seed:27 - Random seed set to 42
2026-09-20 19:44:30.841 | INFO     | tiger.utils:get_device:16 - Using device: mps


Device: mps


Load sid_to_asin mapping

In [4]:
sid_data = torch.load(SEMANTIC_IDS_PATH, weights_only=False)
sid_to_asin = sid_data["sid_to_asin"]

print(f"Items with Semantic IDs: {len(sid_to_asin):,}")

Items with Semantic IDs: 11,924


Load trained model from checkpoint

In [5]:
model = T5ForConditionalGeneration.from_pretrained(CHECKPOINT_PATH).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

Loading weights:   0%|          | 0/89 [00:00<?, ?it/s]

Total parameters: 14,540,160


Create datasets 

In [6]:
val_dataset = TigerDataset(
    splits_path=SPLITS_PATH,
    semantic_ids_path=SEMANTIC_IDS_PATH,
    split="val",
    max_seq_len=20,
)

test_dataset = TigerDataset(
    splits_path=SPLITS_PATH,
    semantic_ids_path=SEMANTIC_IDS_PATH,
    split="test",
    max_seq_len=20,
)

Create evaluator

In [9]:
collate_fn = partial(custom_collate, pad_token_id=val_dataset.pad_token)

retrieval_metrics_fn = partial(
    compute_metrics,
    num_levels=val_dataset.num_levels,
    beam_size=model.generation_config.num_return_sequences,
    codebook_size=val_dataset.codebook_size,
    sid_to_asin=sid_to_asin,
    at_k=AT_K,
)

evaluation_args = Seq2SeqTrainingArguments(
    output_dir="../checkpoints/evaluation",
    per_device_eval_batch_size=BATCH_SIZE,
    predict_with_generate=True,
    remove_unused_columns=False,
    train_sampling_strategy="group_by_length",
    dataloader_pin_memory=device.type == "cuda",
    seed=SEED,
    report_to="none",
)

evaluator = Seq2SeqTrainer(
    model=model,
    args=evaluation_args,
    data_collator=collate_fn,
    compute_metrics=retrieval_metrics_fn,
)

metric_names = [
    f"{name}@{k}"
    for name in METRICS
    for k in AT_K
]

Run small subset for testing

In [10]:
check_results = evaluator.evaluate(
    eval_dataset=Subset(val_dataset, range(129)),
)

for name in metric_names:
    print(f"{name}: {check_results[f'eval_{name}']:.4f}")

Training Loss,Validation Loss,Step,Recall@5,Recall@10,Ndcg@5,Ndcg@10
No log,8.615322,0,0.000000,0.000000,0.000000,0.000000


recall@5: 0.0000
recall@10: 0.0000
ndcg@5: 0.0000
ndcg@10: 0.0000


Run full evaluation on val

In [ ]:
val_output = evaluator.evaluate(
    eval_dataset=val_dataset,
    metric_key_prefix="val",
)

val_results = {
    name: val_output[f"val_{name}"]
    for name in metric_names
}

In [ ]:
test_output = evaluator.evaluate(
    eval_dataset=test_dataset,
    metric_key_prefix="test",
)

test_results = {
    name: test_output[f"test_{name}"]
    for name in metric_names
}

### Compare with Paper (Toys & Games, Table 1)

In [9]:
import pandas as pd

paper = {
    "recall@5": 0.0521,
    "ndcg@5": 0.0371,
    "recall@10": 0.0712,
    "ndcg@10": 0.0432,
}

paper_seeds = {
    "recall@5": (0.0518, 0.00064),
    "ndcg@5": (0.0375, 0.00039),
    "recall@10": (0.0698, 0.0013),
    "ndcg@10": (0.0433, 0.00047),
}

sasrec_best_baseline = {
    "recall@5": 0.0463,
    "ndcg@5": 0.0306,
    "recall@10": 0.0675,
    "ndcg@10": 0.0374,
}

df = pd.DataFrame(
    {
        "Ours (val)": val_results,
        "Ours (test)": test_results,
        "Paper TIGER": paper,
        "Paper (3 seeds)": {
            k: f"{m:.4f} ± {s:.4f}" for k, (m, s) in paper_seeds.items()
        },
        "SASRec baseline": sasrec_best_baseline,
    }
)
df.index.name = "Metric"
df.round(4)

,Ours (val),Ours (test),Paper TIGER,Paper (3 seeds),SASRec baseline
Metric,,,,,
recall@5,0.0299,0.0206,0.0521,0.0518 ± 0.0006,0.0463
ndcg@5,0.0190,0.0127,0.0371,0.0375 ± 0.0004,0.0306
recall@10,0.0478,0.0335,0.0712,0.0698 ± 0.0013,0.0675
ndcg@10,0.0248,0.0168,0.0432,0.0433 ± 0.0005,0.0374
